# LeetCode 25 — Reverse Nodes in k-Group (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def reverseKGroup(self, head: Optional[ListNode], k: int) -> Optional[ListNode]:

```

Mục tiêu sau khi học xong:

1. Hiểu **linked list** (danh sách liên kết) là gì, tự định nghĩa `ListNode`, biết duyệt và in ra.
2. Thuộc lòng kỹ thuật **đảo ngược linked list** bằng 3 con trỏ `prev / curr / nxt`, có hình vẽ từng bước.
3. Đi qua 4 cách giải từ dễ đến tối ưu: chép giá trị → stack → đệ quy → lặp với `dummy` + `group_prev` (bộ nhớ `O(1)`).
4. Trace (mô phỏng) kỹ từng nhóm `k` node để thấy con trỏ thay đổi ra sao.
5. Nộp đúng format LeetCode, biết các lỗi hay gặp và tự giải được các biến thể như Swap Nodes in Pairs, Reverse Linked List II.

## 1. Bài toán yêu cầu gì?

Đề LeetCode (dịch sát nghĩa):

> Cho `head` là node đầu của một linked list. Hãy **đảo ngược các node theo từng nhóm `k` node một**, rồi trả về linked list sau khi đã sửa.
>
> `k` là số nguyên dương và **nhỏ hơn hoặc bằng** độ dài linked list. Nếu số node không chia hết cho `k` thì các node **thừa ở cuối giữ nguyên thứ tự**.
>
> Bạn **không được thay đổi giá trị** (`val`) trong các node, chỉ được thay đổi chính các node (tức là nối lại con trỏ).

**Ví dụ 1:**

```text
Input:  head = [1,2,3,4,5], k = 2
Output: [2,1,4,3,5]
```

**Ví dụ 2:**

```text
Input:  head = [1,2,3,4,5], k = 3
Output: [3,2,1,4,5]
```

**Constraints:**

- Số node trong danh sách là `n`.
- `1 <= k <= n <= 5000`
- `0 <= Node.val <= 1000`

**Follow-up:** Bạn có giải được với bộ nhớ phụ `O(1)` không?

### Làm tay trước khi code

Chia danh sách thành từng nhóm `k` phần tử, đảo **bên trong** mỗi nhóm đủ `k`, nhóm cuối thiếu thì để nguyên:

| Input | k | Các nhóm | Sau khi đảo từng nhóm | Output |
|---|:---:|---|---|---|
| `[1,2,3,4,5]` | 2 | `[1,2] [3,4] [5]` | `[2,1] [4,3] [5]` (nhóm `[5]` thiếu → giữ) | `[2,1,4,3,5]` |
| `[1,2,3,4,5]` | 3 | `[1,2,3] [4,5]` | `[3,2,1] [4,5]` (nhóm `[4,5]` thiếu → giữ) | `[3,2,1,4,5]` |
| `[1,2,3,4,5,6,7,8]` | 3 | `[1,2,3] [4,5,6] [7,8]` | `[3,2,1] [6,5,4] [7,8]` | `[3,2,1,6,5,4,7,8]` |
| `[1,2,3]` | 1 | `[1] [2] [3]` | không đổi gì | `[1,2,3]` |
| `[1,2,3]` | 3 | `[1,2,3]` | `[3,2,1]` | `[3,2,1]` (đảo toàn bộ) |

Ta dễ làm được việc này với **list Python**. Cái khó của bài là dữ liệu nằm trong **linked list**, và đề bắt ta **nối lại con trỏ** chứ không được đổi giá trị. Vì vậy trước hết phải học linked list từ đầu.

## 2. Kiến thức nền: Linked list là gì?

**Linked list** (danh sách liên kết đơn — *singly linked list*) là một chuỗi các **node** (nút). Mỗi node chứa 2 thứ:

- `val`: giá trị (ví dụ số `1`).
- `next`: **con trỏ** (*pointer*, trong Python là một tham chiếu tới object) trỏ sang node kế tiếp. Node cuối có `next = None`.

Hình vẽ danh sách `[1, 2, 3]`:

```text
 head
  |
  v
+---+---+     +---+---+     +---+---+
| 1 | o-+---->| 2 | o-+---->| 3 | o-+----> None
+---+---+     +---+---+     +---+---+
 val next      val next      val next
```

Khác với list Python:

| | List Python `[1,2,3]` | Linked list |
|---|---|---|
| Truy cập phần tử thứ `i` | `a[i]` — nhảy thẳng tới | Phải đi từ `head`, bước `i` lần theo `next` |
| Ta cầm trong tay | cả mảng | chỉ **node đầu** `head` |
| Đảo đoạn giữa | cắt/ghép mảng | **nối lại** các mũi tên `next` |

Trên LeetCode, lớp `ListNode` đã được định nghĩa sẵn. Ở notebook này ta tự viết lại (thêm `__repr__` để in cho dễ nhìn).

In [1]:
from typing import Optional


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    def __repr__(self):
        return f"ListNode({self.val})"


# Tạo tay danh sách 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a

print(head)            # node đầu
print(head.val)        # giá trị node đầu
print(head.next)       # node thứ hai
print(head.next.next)  # node thứ ba
print(head.next.next.next)  # hết danh sách -> None

ListNode(1)
1
ListNode(2)
ListNode(3)
None


### Duyệt linked list: `curr = curr.next`

Muốn đi qua mọi node, ta dùng một biến `curr` (*current* — node hiện tại), bắt đầu ở `head`, mỗi vòng bước sang `curr.next`, tới khi gặp `None` thì dừng:

```python
curr = head
while curr is not None:
    ...            # xử lý curr.val
    curr = curr.next
```

Chú ý: ta dùng biến **phụ** `curr` để đi, giữ nguyên `head` để không "mất" đầu danh sách.

In [2]:
curr = head
step = 0
while curr is not None:
    print(f"bước {step}: curr = {curr}, curr.val = {curr.val}, curr.next = {curr.next}")
    curr = curr.next
    step += 1
print("curr cuối cùng:", curr)

bước 0: curr = ListNode(1), curr.val = 1, curr.next = ListNode(2)
bước 1: curr = ListNode(2), curr.val = 2, curr.next = ListNode(3)
bước 2: curr = ListNode(3), curr.val = 3, curr.next = None
curr cuối cùng: None


### Hàm tiện ích: `build_list`, `to_list`, `show`

Gõ tay từng node rất mỏi, nên ta viết 3 helper dùng suốt notebook:

- `build_list([1,2,3])` → dựng linked list, trả về `head`.
- `to_list(head)` → đổi ngược ra list Python `[1,2,3]` để so sánh bằng `assert`. Có chốt an toàn: nếu đi quá nhiều bước (dấu hiệu danh sách bị **vòng lặp** — *cycle*, do nối sai con trỏ) thì báo lỗi thay vì chạy mãi.
- `show(head)` → chuỗi đẹp `"1 -> 2 -> 3 -> None"`; nếu quá dài/vòng lặp thì in `...`.

`build_list` dùng mẹo **dummy node** (node giả đứng trước đầu) — ta sẽ giải thích kỹ ở mục 4.

In [3]:
def build_list(values) -> Optional[ListNode]:
    dummy = ListNode(0)
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode], limit: int = 100_000) -> list:
    out = []
    curr = head
    while curr is not None:
        out.append(curr.val)
        if len(out) > limit:
            raise RuntimeError("Đi quá nhiều bước: có vẻ linked list bị vòng lặp (cycle)!")
        curr = curr.next
    return out


def show(head: Optional[ListNode], limit: int = 15) -> str:
    parts = []
    curr = head
    while curr is not None and len(parts) < limit:
        parts.append(str(curr.val))
        curr = curr.next
    if curr is not None:
        return " -> ".join(parts) + " -> ... (vòng lặp?)"
    return " -> ".join(parts + ["None"])


h = build_list([1, 2, 3, 4, 5])
print(show(h))
print(to_list(h))
print(show(build_list([])))   # danh sách rỗng: head = None

1 -> 2 -> 3 -> 4 -> 5 -> None
[1, 2, 3, 4, 5]
None


### "Không đổi value, chỉ đổi node" nghĩa là gì?

Hai cách để danh sách in ra `2 -> 1`:

1. **Đổi value** (bị cấm): giữ nguyên mũi tên, đổi `a.val` và `b.val` cho nhau.
2. **Đổi node / con trỏ** (đúng đề): giữ nguyên value, **nối lại** mũi tên `next`, để `b` đứng trước `a`.

```text
Ban đầu:      head -> [a:1] -> [b:2] -> None

Nối lại:      b.next = a ; a.next = None ; head = b
Kết quả:      head -> [b:2] -> [a:1] -> None     (object a, b vẫn là chính chúng)
```

Trong Python, `x is y` kiểm tra **cùng một object** (cùng một node), khác với `x == y`. Ta sẽ dùng `is` để chứng minh lời giải thật sự dùng lại node cũ.

In [4]:
a = ListNode(1)
b = ListNode(2)
a.next = b
head = a
print("Trước:", show(head))

# Đổi node: nối lại con trỏ
b.next = a
a.next = None
head = b
print("Sau  :", show(head))
print("head is b ?", head is b, "| head.next is a ?", head.next is a)
print("a.val vẫn là", a.val, "| b.val vẫn là", b.val)

Trước: 1 -> 2 -> None
Sau  : 2 -> 1 -> None
head is b ? True | head.next is a ? True
a.val vẫn là 1 | b.val vẫn là 2


## 3. Kỹ thuật nền quan trọng nhất: đảo ngược cả linked list (`prev / curr / nxt`)

Đây chính là bài LeetCode 206 — Reverse Linked List. Bài 25 chỉ là "đảo ngược từng khúc" nên phải nắm chắc phần này.

Ý tưởng: đi từ đầu tới cuối, **lật mũi tên `next` của từng node quay ngược lại**. Cần 3 biến:

- `prev`: phần **đã đảo xong** (lúc đầu chưa có gì → `None`).
- `curr`: node **đang xử lý**.
- `nxt`: **lưu tạm** `curr.next` trước khi lật, nếu không sẽ mất đường đi tiếp.

Mỗi vòng có đúng 4 dòng, thứ tự **không được đảo**:

```python
nxt = curr.next     # 1. giữ đường đi tiếp
curr.next = prev    # 2. lật mũi tên của curr về phía sau
prev = curr         # 3. prev tiến lên
curr = nxt          # 4. curr tiến lên
```

### Hình vẽ từng bước với `[1, 2, 3]`

```text
Bắt đầu:   prev = None     curr
                            |
                            v
                           [1] -> [2] -> [3] -> None

Vòng 1:    nxt = [2];  1.next = None;  prev = [1];  curr = [2]
           None <- [1]     [2] -> [3] -> None
                    ^       ^
                   prev    curr

Vòng 2:    nxt = [3];  2.next = [1];   prev = [2];  curr = [3]
           None <- [1] <- [2]     [3] -> None
                           ^       ^
                          prev    curr

Vòng 3:    nxt = None; 3.next = [2];   prev = [3];  curr = None
           None <- [1] <- [2] <- [3]     None
                                  ^       ^
                                 prev    curr  -> dừng vòng lặp
```

Kết thúc: `prev` chính là **đầu mới** (`3 -> 2 -> 1 -> None`). Node `[1]` (đầu cũ) giờ là **đuôi mới**.

Hai nửa luôn là **hai linked list rời nhau** hợp lệ: `prev` (đã đảo) và `curr` (chưa đảo). Hàm trace dưới đây in đúng hai nửa đó sau mỗi vòng.

In [5]:
def reverse_list(head: Optional[ListNode]) -> Optional[ListNode]:
    prev = None
    curr = head
    while curr is not None:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    return prev


def trace_reverse_list(values):
    prev, curr = None, build_list(values)
    print(f"bắt đầu | prev: {show(prev):<22} | curr: {show(curr)}")
    step = 1
    while curr is not None:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
        print(f"vòng {step}  | prev: {show(prev):<22} | curr: {show(curr)}")
        step += 1
    return prev


new_head = trace_reverse_list([1, 2, 3, 4])
print("Kết quả:", to_list(new_head))
assert to_list(reverse_list(build_list([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert to_list(reverse_list(build_list([7]))) == [7]
assert reverse_list(None) is None

bắt đầu | prev: None                   | curr: 1 -> 2 -> 3 -> 4 -> None
vòng 1  | prev: 1 -> None              | curr: 2 -> 3 -> 4 -> None
vòng 2  | prev: 2 -> 1 -> None         | curr: 3 -> 4 -> None
vòng 3  | prev: 3 -> 2 -> 1 -> None    | curr: 4 -> None
vòng 4  | prev: 4 -> 3 -> 2 -> 1 -> None | curr: None
Kết quả: [4, 3, 2, 1]


### Mở rộng nhỏ: chỉ đảo `k` node đầu

Bài 25 cần đảo **một khúc** chứ không phải cả danh sách. Thay `while curr is not None` bằng "lặp đúng `k` lần" là xong. Sau `k` vòng:

- `prev` = node thứ `k` cũ = **đầu mới** của khúc.
- `head` (node đầu cũ) = **đuôi mới** của khúc, hiện đang có `next = None` (bị đứt khỏi phần còn lại!).
- `curr` = node thứ `k+1` = **đầu phần chưa xử lý**.

Muốn danh sách liền mạch, phải nối lại: `head.next = curr`.

```text
k = 3, trước:   [1] -> [2] -> [3] -> [4] -> [5] -> None

sau 3 vòng:     None <- [1] <- [2] <- [3]      [4] -> [5] -> None
                         ^             ^        ^
                     đuôi mới       prev      curr
                    (head cũ)    (đầu mới)

nối head.next = curr:   [3] -> [2] -> [1] -> [4] -> [5] -> None
```

In [6]:
def reverse_first_k(head: ListNode, k: int):
    # Giả sử danh sách có ít nhất k node.
    prev, curr = None, head
    for _ in range(k):
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    # prev: đầu mới, head: đuôi mới, curr: đầu phần còn lại
    return prev, head, curr


h = build_list([1, 2, 3, 4, 5])
new_head, new_tail, rest = reverse_first_k(h, 3)
print("khúc đã đảo:", show(new_head))
print("phần còn lại:", show(rest))
new_tail.next = rest          # nối lại cho liền mạch
print("sau khi nối :", show(new_head))

khúc đã đảo: 3 -> 2 -> 1 -> None
phần còn lại: 4 -> 5 -> None
sau khi nối : 3 -> 2 -> 1 -> 4 -> 5 -> None


## 4. Kiến thức nền: dummy node (node giả đứng trước `head`)

Khi đảo nhóm **đầu tiên**, `head` của cả danh sách bị thay đổi (ví dụ từ `1` thành `3`). Với các nhóm sau, ta lại phải sửa `next` của **node đứng trước nhóm**. Nhóm đầu tiên không có "node đứng trước" → phải viết `if` riêng, rất dễ sai.

Mẹo: tạo một node giả `dummy` có `dummy.next = head`. Giờ **mọi nhóm đều có node đứng trước**, kể cả nhóm đầu. Cuối cùng trả về `dummy.next`.

```text
dummy -> [1] -> [2] -> [3] -> [4] -> [5] -> None
  ^
"node đứng trước" của nhóm đầu tiên
```

Ví dụ mini: xóa node đầu tiên có giá trị `x`. Không có dummy thì phải xét riêng trường hợp `x` nằm ở `head`; có dummy thì một đoạn code xử lý hết.

In [7]:
def remove_first(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy                      # luôn là node đứng TRƯỚC node đang xét
    while prev.next is not None:
        if prev.next.val == x:
            prev.next = prev.next.next   # bỏ qua node cần xóa
            break
        prev = prev.next
    return dummy.next                 # head mới (có thể đã đổi)


print(show(remove_first(build_list([1, 2, 3]), 1)))  # xóa ở đầu: head đổi
print(show(remove_first(build_list([1, 2, 3]), 2)))  # xóa ở giữa
print(show(remove_first(build_list([1]), 1)))        # xóa hết

2 -> 3 -> None
1 -> 3 -> None
None


## 5. Cách 1 — Chép giá trị ra list Python, đảo từng đoạn, dựng lại (baseline)

Cách người mới hay nghĩ đầu tiên: "linked list khó quá, đổi về list Python cho dễ".

1. Duyệt linked list, chép các `val` ra list `vals`.
2. Với mỗi đoạn `vals[i : i+k]` **đủ `k` phần tử**, đảo bằng slicing `[::-1]`.
3. Dựng một linked list **mới** từ `vals`.

Mini nhắc lại slicing: `vals[i:i+k]` lấy đoạn từ `i` tới trước `i+k`; `[::-1]` đảo ngược đoạn đó; gán `vals[i:i+k] = ...` thay đoạn cũ bằng đoạn mới.

In [8]:
vals = [1, 2, 3, 4, 5]
print(vals[0:2], "->", vals[0:2][::-1])
vals[0:2] = vals[0:2][::-1]
print(vals)

[1, 2] -> [2, 1]
[2, 1, 3, 4, 5]


In [9]:
def reverse_k_group_values(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    vals = to_list(head)                     # 1. chép giá trị
    for i in range(0, len(vals) - k + 1, k):  # 2. chỉ các đoạn đủ k phần tử
        vals[i:i + k] = vals[i:i + k][::-1]
    return build_list(vals)                  # 3. dựng danh sách MỚI


for values, k in [([1, 2, 3, 4, 5], 2), ([1, 2, 3, 4, 5], 3), ([1, 2, 3, 4, 5, 6, 7, 8], 3)]:
    print(values, "k =", k, "->", to_list(reverse_k_group_values(build_list(values), k)))

[1, 2, 3, 4, 5] k = 2 -> [2, 1, 4, 3, 5]
[1, 2, 3, 4, 5] k = 3 -> [3, 2, 1, 4, 5]
[1, 2, 3, 4, 5, 6, 7, 8] k = 3 -> [3, 2, 1, 6, 5, 4, 7, 8]


### Vì sao `range(0, len(vals) - k + 1, k)`?

`i` là vị trí bắt đầu mỗi nhóm: `0, k, 2k, ...`. Nhóm bắt đầu ở `i` đủ `k` phần tử khi `i + k <= len(vals)`, tức `i <= len(vals) - k`. `range` dừng **trước** cận trên nên viết `len(vals) - k + 1`. Ví dụ `len = 5, k = 3`: `range(0, 3, 3)` chỉ cho `i = 0` → nhóm `[4,5]` không bị đảo. Đúng ý đề.

### Cách 1 vướng ở đâu?

- **Không đúng tinh thần đề:** đề nói *"You may not alter the values in the list's nodes, only nodes themselves may be changed"*. Cách này **tạo node mới** (hoặc nếu ghi đè lại `val` vào node cũ thì là **đổi value**) — cả hai đều không phải "nối lại node". Trong phỏng vấn, cách này coi như chưa giải.
- **Bộ nhớ `O(n)`** cho list `vals` và danh sách mới, trái với follow-up `O(1)`.

**Độ phức tạp Cách 1** (gọi `n` là số node):

- Thời gian: `O(n)` — chép 1 lần, đảo tổng cộng `n` phần tử, dựng lại 1 lần.
- Bộ nhớ: `O(n)` — list `vals` + `n` node mới.

Cách 1 vẫn có ích: nó là **đáp án chuẩn** để đối chiếu các cách khác trên input ngẫu nhiên (mục 9).

Ta kiểm chứng bằng `is`: node đầu của kết quả **không phải** node cũ nào cả.

In [10]:
h = build_list([1, 2, 3, 4, 5])
old_nodes = []
curr = h
while curr is not None:
    old_nodes.append(curr)
    curr = curr.next

res = reverse_k_group_values(h, 2)
print("Kết quả:", show(res))
print("res có phải một node cũ không?", any(res is node for node in old_nodes))

Kết quả: 2 -> 1 -> 4 -> 3 -> 5 -> None
res có phải một node cũ không? False


## 6. Cách 2 — Dùng stack chứa `k` node

### Kiến thức nền: stack là gì?

**Stack** (ngăn xếp) giống chồng đĩa: đặt đĩa lên trên cùng (**push**) và lấy đĩa trên cùng ra (**pop**). Vào sau ra trước — *LIFO: Last In, First Out*.

Trong Python, list chính là stack: `append` = push, `pop()` = pop phần tử cuối. Tính chất quan trọng cho bài này: **push theo thứ tự 1, 2, 3 thì pop ra theo thứ tự 3, 2, 1** — tức là tự động đảo ngược!

In [11]:
stack = []
for x in [1, 2, 3]:
    stack.append(x)
    print("push", x, "-> stack =", stack)

while stack:
    print("pop ", stack.pop(), "-> stack =", stack)

push 1 -> stack = [1]
push 2 -> stack = [1, 2]
push 3 -> stack = [1, 2, 3]
pop  3 -> stack = [1, 2]
pop  2 -> stack = [1]
pop  1 -> stack = []


### Ý tưởng Cách 2

Khác Cách 1: ta đẩy **chính các node** (không phải value) vào stack, rồi nối lại con trỏ.

1. Tạo `dummy`, biến `tail = dummy` là **đuôi của phần kết quả đã xếp xong**.
2. Đẩy tối đa `k` node tiếp theo vào stack (`curr` đi tới node sau đó).
3. Nếu stack **đủ `k` node**: pop lần lượt và nối vào `tail` → nhóm được đảo.
4. Nếu **thiếu**: đây là phần thừa cuối → nối `tail.next` vào node đầu của phần thừa (giữ nguyên thứ tự) rồi dừng.
5. Đừng quên `tail.next = curr` sau mỗi nhóm để đuôi không trỏ lung tung (đặc biệt khi hết danh sách phải về `None`).

Ưu điểm so với Cách 1: **dùng lại node cũ, không đổi value** → đúng đề. Nhưng vẫn tốn thêm bộ nhớ cho stack.

In [12]:
def reverse_k_group_stack(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    dummy = ListNode(0)
    tail = dummy
    curr = head
    while True:
        stack = []
        group_start = curr
        while curr is not None and len(stack) < k:
            stack.append(curr)
            curr = curr.next
        if len(stack) < k:            # nhóm thiếu: giữ nguyên thứ tự
            tail.next = group_start
            break
        while stack:                  # pop ra theo thứ tự ngược
            tail.next = stack.pop()
            tail = tail.next
        tail.next = curr              # tạm nối với phần còn lại (an toàn khi curr = None)
    return dummy.next


for values, k in [([1, 2, 3, 4, 5], 2), ([1, 2, 3, 4, 5], 3), ([1, 2, 3], 3)]:
    print(values, "k =", k, "->", to_list(reverse_k_group_stack(build_list(values), k)))

[1, 2, 3, 4, 5] k = 2 -> [2, 1, 4, 3, 5]
[1, 2, 3, 4, 5] k = 3 -> [3, 2, 1, 4, 5]
[1, 2, 3] k = 3 -> [3, 2, 1]


### Trace Cách 2 với `[1,2,3,4,5,6,7,8]`, `k = 3`

| Nhóm | Stack sau khi push | Đủ k? | Hành động | Kết quả đã xếp |
|:---:|---|:---:|---|---|
| 1 | `[1, 2, 3]` | có | pop 3, 2, 1 nối vào `tail` | `3 -> 2 -> 1` |
| 2 | `[4, 5, 6]` | có | pop 6, 5, 4 | `3 -> 2 -> 1 -> 6 -> 5 -> 4` |
| 3 | `[7, 8]` | không | nối `tail.next = 7`, dừng | `3 -> 2 -> 1 -> 6 -> 5 -> 4 -> 7 -> 8` |

In [13]:
def trace_stack(values, k):
    dummy = ListNode(0)
    tail = dummy
    curr = build_list(values)
    group = 1
    while True:
        stack = []
        group_start = curr
        while curr is not None and len(stack) < k:
            stack.append(curr)
            curr = curr.next
        print(f"Nhóm {group}: stack = {[node.val for node in stack]}", end=" ")
        if len(stack) < k:
            tail.next = group_start
            print(f"-> thiếu ({len(stack)} < {k}), giữ nguyên")
            break
        popped = []
        while stack:
            tail.next = stack.pop()
            tail = tail.next
            popped.append(tail.val)
        tail.next = curr
        print(f"-> pop {popped}")
        group += 1
    print("Kết quả:", show(dummy.next))


trace_stack([1, 2, 3, 4, 5, 6, 7, 8], 3)

Nhóm 1: stack = [1, 2, 3] -> pop [3, 2, 1]
Nhóm 2: stack = [4, 5, 6] -> pop [6, 5, 4]
Nhóm 3: stack = [7, 8] -> thiếu (2 < 3), giữ nguyên
Kết quả: 3 -> 2 -> 1 -> 6 -> 5 -> 4 -> 7 -> 8 -> None


**Độ phức tạp Cách 2:**

- Thời gian: `O(n)` — mỗi node được push 1 lần, pop tối đa 1 lần.
- Bộ nhớ: `O(k)` cho stack (không tính kết quả vì dùng lại node cũ).

**Còn vướng ở đâu?** Khi `k` lớn (ví dụ `k = n = 5000`), stack chứa cả `n` node → vẫn `O(n)`. Follow-up muốn `O(1)`. Mấu chốt cải tiến: **đảo tại chỗ** bằng `prev / curr / nxt` (mục 3) thay vì nhờ stack đảo hộ.

## 7. Cách 3 — Đệ quy: đảo `k` node đầu, phần sau "nhờ" chính hàm mình

### Kiến thức nền: recursion (đệ quy)

**Đệ quy** là khi một hàm **gọi lại chính nó** trên bài toán **nhỏ hơn**. Luôn cần 2 phần:

1. **Base case** (trường hợp dừng): bài đủ nhỏ để trả lời ngay.
2. **Bước đệ quy**: làm một phần việc, phần còn lại giao cho lời gọi đệ quy.

Ví dụ mini: độ dài linked list = `1 + độ dài phần sau head`; danh sách rỗng có độ dài `0`. Mỗi lời gọi được đặt lên **call stack** (ngăn xếp lời gọi) cho tới khi gặp base case, rồi lần lượt trả kết quả ngược lên.

In [14]:
def length_recursive(node, depth=0):
    indent = "  " * depth
    if node is None:                       # base case
        print(f"{indent}length(None) = 0")
        return 0
    print(f"{indent}length({node.val}) = 1 + length(phần sau)")
    ans = 1 + length_recursive(node.next, depth + 1)
    print(f"{indent}=> length({node.val}) = {ans}")
    return ans


_ = length_recursive(build_list([1, 2, 3]))

length(1) = 1 + length(phần sau)
  length(2) = 1 + length(phần sau)
    length(3) = 1 + length(phần sau)
      length(None) = 0
    => length(3) = 1
  => length(2) = 2
=> length(1) = 3


### Ý tưởng Cách 3

Định nghĩa `reverse_k_group_recursive(head, k)` = "trả về đầu mới của danh sách bắt đầu từ `head` sau khi đảo theo nhóm `k`".

1. **Đếm** xem từ `head` có đủ `k` node không. Nếu **không đủ** → base case: trả `head` nguyên vẹn (phần thừa giữ nguyên).
2. Nếu đủ: **đảo `k` node đầu** bằng `prev / curr / nxt` (như `reverse_first_k`). Sau đó `prev` là đầu mới, `head` thành đuôi của nhóm, `curr` là đầu phần sau.
3. **Nối** đuôi nhóm với kết quả đệ quy của phần sau: `head.next = reverse_k_group_recursive(curr, k)`.
4. Trả về `prev`.

```text
[1 2 3] [4 5 6 7 8]          k = 3
   |         |
 đảo tay   giao cho đệ quy  ->  6 -> 5 -> 4 -> 7 -> 8
   v
[3 2 1] -> (kết quả đệ quy)
```

In [15]:
def reverse_k_group_recursive(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    # 1. Kiểm tra đủ k node
    node, count = head, 0
    while node is not None and count < k:
        node = node.next
        count += 1
    if count < k:
        return head                       # base case: nhóm thiếu, giữ nguyên

    # 2. Đảo k node đầu
    prev, curr = None, head
    for _ in range(k):
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt

    # 3. head giờ là đuôi nhóm; nối với kết quả đệ quy của phần sau
    head.next = reverse_k_group_recursive(curr, k)
    return prev                           # 4. đầu mới của nhóm


for values, k in [([1, 2, 3, 4, 5], 2), ([1, 2, 3, 4, 5], 3), ([1, 2, 3, 4, 5, 6, 7, 8], 3)]:
    print(values, "k =", k, "->", to_list(reverse_k_group_recursive(build_list(values), k)))

[1, 2, 3, 4, 5] k = 2 -> [2, 1, 4, 3, 5]
[1, 2, 3, 4, 5] k = 3 -> [3, 2, 1, 4, 5]
[1, 2, 3, 4, 5, 6, 7, 8] k = 3 -> [3, 2, 1, 6, 5, 4, 7, 8]


### Trace Cách 3 — nhìn call stack

Hàm trace in thụt lề theo độ sâu lời gọi: đi xuống thì "đảo nhóm, giao phần sau", gặp base case thì đi lên và nối.

In [16]:
def trace_recursive(head, k, depth=0):
    indent = "    " * depth
    node, count = head, 0
    while node is not None and count < k:
        node = node.next
        count += 1
    if count < k:
        print(f"{indent}gọi({show(head)}): chỉ {count} node < k -> trả nguyên")
        return head
    prev, curr = None, head
    for _ in range(k):
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    print(f"{indent}gọi: đảo nhóm thành {show(prev)}, giao phần sau cho đệ quy")
    head.next = trace_recursive(curr, k, depth + 1)
    print(f"{indent}nối đuôi {head.val} -> kết quả: {show(prev)}")
    return prev


_ = trace_recursive(build_list([1, 2, 3, 4, 5, 6, 7, 8]), 3)

gọi: đảo nhóm thành 3 -> 2 -> 1 -> None, giao phần sau cho đệ quy
    gọi: đảo nhóm thành 6 -> 5 -> 4 -> None, giao phần sau cho đệ quy
        gọi(7 -> 8 -> None): chỉ 2 node < k -> trả nguyên
    nối đuôi 4 -> kết quả: 6 -> 5 -> 4 -> 7 -> 8 -> None
nối đuôi 1 -> kết quả: 3 -> 2 -> 1 -> 6 -> 5 -> 4 -> 7 -> 8 -> None


**Độ phức tạp Cách 3:**

- Thời gian: `O(n)` — mỗi node được đếm 1 lần và đảo 1 lần (tổng cộng khoảng `2n` bước).
- Bộ nhớ: `O(n / k)` cho call stack (mỗi nhóm một lời gọi).

**Vướng ở đâu?** Call stack **cũng là bộ nhớ**. Khi `k = 1` và `n = 5000` (vẫn đúng constraints), độ sâu đệ quy là `5000`. Python chuẩn giới hạn độ sâu đệ quy (xem `sys.getrecursionlimit()`), vượt quá sẽ báo `RecursionError`. Thử ngay:

In [17]:
import sys

print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
big = build_list(range(5000))
try:
    reverse_k_group_recursive(big, 1)
    print("Chạy được")
except RecursionError as e:
    print("RecursionError:", e)

Giới hạn đệ quy hiện tại: 1000
RecursionError: maximum recursion depth exceeded


Có thể "chữa cháy" bằng `sys.setrecursionlimit(...)`, nhưng đó không phải lời giải `O(1)` bộ nhớ. Mấu chốt cải tiến cuối cùng: **biến đệ quy thành vòng lặp** — thay vì "giao phần sau cho lời gọi khác rồi nối khi quay về", ta **tự nhớ node đứng trước nhóm** (`group_prev`) và nối ngay khi đảo xong mỗi nhóm.

## 8. Cách 4 (cách chính) — Lặp với `dummy` + `group_prev`, bộ nhớ `O(1)`

### Các con trỏ cần dùng

Ở mỗi nhóm ta cầm 4 node quan trọng:

```text
 group_prev     nhóm hiện tại (k node)       group_next
     |       +---------------------------+       |
     v       v                           v       v
   [ P ] -> [ A ] -> [ B ] -> ... -> [ kth ] -> [ N ] -> ...
```

- `group_prev`: node **đứng ngay trước** nhóm (lần đầu là `dummy`).
- `kth`: node **thứ k** của nhóm, tính từ `group_prev` đi `k` bước. Nếu đi chưa đủ `k` bước đã gặp `None` → nhóm thiếu → **dừng**.
- `group_next = kth.next`: node **đầu nhóm sau**.
- `A = group_prev.next`: node đầu nhóm (sau khi đảo sẽ thành **đuôi** nhóm).

Mục tiêu sau khi xử lý một nhóm:

```text
   [ P ] -> [ kth ] -> ... -> [ B ] -> [ A ] -> [ N ] -> ...
                                         ^
                                 group_prev mới = A
```

### Bước 1: helper `get_kth(curr, k)` — đi `k` bước

Hàm đi `k` bước theo `next` từ `curr`; nếu giữa chừng gặp `None` thì trả `None`. Nhờ vậy ta **kiểm tra đủ `k` node trước khi đảo** — không bao giờ lỡ tay đảo nhóm thiếu.

In [18]:
def get_kth(curr: Optional[ListNode], k: int) -> Optional[ListNode]:
    while curr is not None and k > 0:
        curr = curr.next
        k -= 1
    return curr


dummy = ListNode(0, build_list([1, 2, 3, 4, 5]))
print("đi 3 bước từ dummy:", get_kth(dummy, 3))       # node 3
print("đi 5 bước từ dummy:", get_kth(dummy, 5))       # node 5
print("đi 6 bước từ dummy:", get_kth(dummy, 6))       # None: không đủ
node3 = get_kth(dummy, 3)
print("đi 3 bước từ node 3:", get_kth(node3, 3))      # None: nhóm [4,5] thiếu

đi 3 bước từ dummy: ListNode(3)
đi 5 bước từ dummy: ListNode(5)
đi 6 bước từ dummy: None
đi 3 bước từ node 3: None


### Bước 2: đảo nhóm tại chỗ với mẹo `prev = group_next`

Ta dùng lại vòng `prev / curr / nxt`, nhưng **khởi tạo `prev = group_next` thay vì `None`**. Vì sao? Node đầu nhóm `A` sẽ trở thành đuôi nhóm; khi lật mũi tên của `A`, nó sẽ trỏ về `prev` ban đầu — tức trỏ thẳng sang `group_next`. Vậy là **đuôi nhóm tự động nối với nhóm sau**, không cần `if` riêng.

Vòng lặp dừng khi `curr` đi tới `group_next` (đã ra khỏi nhóm).

```text
group_prev=P, nhóm [1,2,3], group_next=[4]

bắt đầu:  prev=[4]  curr=[1]       P -> 1 -> 2 -> 3 -> 4
vòng 1:   1.next=[4]; prev=[1]; curr=[2]
vòng 2:   2.next=[1]; prev=[2]; curr=[3]
vòng 3:   3.next=[2]; prev=[3]; curr=[4] = group_next -> dừng

lúc này:  3 -> 2 -> 1 -> 4 -> ...   (nhưng P vẫn đang trỏ vào 1!)
```

### Bước 3: nối `group_prev` vào đầu mới, rồi dời `group_prev`

```python
old_first = group_prev.next   # = A (node 1), giờ là đuôi nhóm
group_prev.next = kth         # P -> 3
group_prev = old_first        # nhóm sau có "node đứng trước" là 1
```

```text
P -> 3 -> 2 -> 1 -> 4 -> 5 -> ...
               ^
         group_prev mới
```

In [19]:
def reverse_k_group_iterative(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    group_prev = dummy

    while True:
        kth = get_kth(group_prev, k)
        if kth is None:                  # không đủ k node -> giữ nguyên phần còn lại
            break
        group_next = kth.next

        # Đảo nhóm tại chỗ; prev bắt đầu ở group_next để đuôi nhóm nối sẵn sang nhóm sau
        prev, curr = group_next, group_prev.next
        while curr is not group_next:
            nxt = curr.next
            curr.next = prev
            prev = curr
            curr = nxt

        # Nối node đứng trước nhóm với đầu mới (kth), rồi dời group_prev tới đuôi mới
        old_first = group_prev.next
        group_prev.next = kth
        group_prev = old_first

    return dummy.next


for values, k in [([1, 2, 3, 4, 5], 2), ([1, 2, 3, 4, 5], 3), ([1, 2, 3, 4, 5, 6, 7, 8], 3)]:
    print(values, "k =", k, "->", to_list(reverse_k_group_iterative(build_list(values), k)))

[1, 2, 3, 4, 5] k = 2 -> [2, 1, 4, 3, 5]
[1, 2, 3, 4, 5] k = 3 -> [3, 2, 1, 4, 5]
[1, 2, 3, 4, 5, 6, 7, 8] k = 3 -> [3, 2, 1, 6, 5, 4, 7, 8]


### Trace kỹ Cách 4 với `[1,2,3,4,5,6,7,8]`, `k = 3`

**Bảng theo nhóm** (`dummy` viết là `D`):

| Nhóm | `group_prev` | `kth` | `group_next` | Các vòng đảo (`prev`, `curr` sau mỗi vòng) | Nối | `group_prev` mới | Danh sách sau nhóm |
|:---:|:---:|:---:|:---:|---|---|:---:|---|
| 1 | `D` | `3` | `4` | bắt đầu (4,1) → (1,2) → (2,3) → (3,4) | `D.next = 3` | `1` | `3,2,1,4,5,6,7,8` |
| 2 | `1` | `6` | `7` | bắt đầu (7,4) → (4,5) → (5,6) → (6,7) | `1.next = 6` | `4` | `3,2,1,6,5,4,7,8` |
| 3 | `4` | `None` | — | không đảo (chỉ còn `7, 8`) | — | — | dừng, trả `D.next` |

Chú ý nhóm 2: sau nhóm 1, node `1` đang trỏ tới `4`. Nếu **quên** dòng `group_prev.next = kth` ở nhóm 2, node `1` vẫn trỏ vào `4` (giờ là đuôi nhóm 2) → mất `6, 5`. Đó là lý do bước nối là bắt buộc.

Hàm trace in đúng các trạng thái trên:

In [20]:
def trace_iterative(values, k):
    dummy = ListNode("D", build_list(values))
    group_prev = dummy
    group = 1
    while True:
        kth = get_kth(group_prev, k)
        if kth is None:
            print(f"Nhóm {group}: group_prev={group_prev.val}, kth=None -> không đủ {k} node, dừng")
            break
        group_next = kth.next
        gn = group_next.val if group_next else None
        print(f"Nhóm {group}: group_prev={group_prev.val}, kth={kth.val}, group_next={gn}")
        prev, curr = group_next, group_prev.next
        print(f"   bắt đầu: prev={gn}, curr={curr.val}")
        step = 1
        while curr is not group_next:
            nxt = curr.next
            curr.next = prev
            prev = curr
            curr = nxt
            cv = curr.val if curr else None
            print(f"   vòng {step}: {prev.val}.next -> {prev.next.val if prev.next else None}"
                  f" | prev={prev.val}, curr={cv}")
            step += 1
        old_first = group_prev.next
        group_prev.next = kth
        group_prev = old_first
        print(f"   nối: next của node trước nhóm -> {kth.val}; group_prev mới = {group_prev.val}")
        print(f"   danh sách: {show(dummy.next)}")
        group += 1
    return dummy.next


res = trace_iterative([1, 2, 3, 4, 5, 6, 7, 8], 3)

Nhóm 1: group_prev=D, kth=3, group_next=4
   bắt đầu: prev=4, curr=1
   vòng 1: 1.next -> 4 | prev=1, curr=2
   vòng 2: 2.next -> 1 | prev=2, curr=3
   vòng 3: 3.next -> 2 | prev=3, curr=4
   nối: next của node trước nhóm -> 3; group_prev mới = 1
   danh sách: 3 -> 2 -> 1 -> 4 -> 5 -> 6 -> 7 -> 8 -> None
Nhóm 2: group_prev=1, kth=6, group_next=7
   bắt đầu: prev=7, curr=4
   vòng 1: 4.next -> 7 | prev=4, curr=5
   vòng 2: 5.next -> 4 | prev=5, curr=6
   vòng 3: 6.next -> 5 | prev=6, curr=7
   nối: next của node trước nhóm -> 6; group_prev mới = 4
   danh sách: 3 -> 2 -> 1 -> 6 -> 5 -> 4 -> 7 -> 8 -> None
Nhóm 3: group_prev=4, kth=None -> không đủ 3 node, dừng


### Hình vẽ toàn cảnh sau từng nhóm

```text
Ban đầu:
  D -> 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> None
  ^
  group_prev

Sau nhóm 1:
  D -> 3 -> 2 -> 1 -> 4 -> 5 -> 6 -> 7 -> 8 -> None
                 ^
                 group_prev

Sau nhóm 2:
  D -> 3 -> 2 -> 1 -> 6 -> 5 -> 4 -> 7 -> 8 -> None
                                ^
                                group_prev

Nhóm 3: get_kth(4, 3) gặp None sau 7, 8  ->  dừng, trả D.next = 3
```

Bất biến dễ thấy trên hình: **mọi node từ `D` tới `group_prev` đã ở đúng vị trí cuối cùng**, và `group_prev.next` là đầu phần chưa xử lý.

## 9. Code nộp LeetCode — Cách 4

Trên LeetCode, `ListNode` và `Optional` đã có sẵn; chỉ cần dán class `Solution`. Ở đây ta dùng `ListNode` đã định nghĩa ở đầu notebook.

In [21]:
class Solution:
    def reverseKGroup(self, head: Optional[ListNode], k: int) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        group_prev = dummy

        while True:
            # tìm node thứ k tính từ group_prev
            kth = group_prev
            for _ in range(k):
                kth = kth.next
                if kth is None:
                    return dummy.next
            group_next = kth.next

            # đảo nhóm tại chỗ
            prev, curr = group_next, group_prev.next
            while curr is not group_next:
                nxt = curr.next
                curr.next = prev
                prev = curr
                curr = nxt

            # nối lại và dời group_prev
            old_first = group_prev.next
            group_prev.next = kth
            group_prev = old_first

### Test: ví dụ đề + edge case + kiểm tra "không đổi value"

Ngoài so sánh output, ta kiểm tra thêm: kết quả phải gồm **đúng các node object cũ** (dùng `id`), và mỗi node vẫn giữ `val` ban đầu.

In [22]:
def nodes_of(head):
    out = []
    while head is not None:
        out.append(head)
        head = head.next
    return out


def check_solution(func, values, k, expected):
    head = build_list(values)
    before = {id(node): node.val for node in nodes_of(head)}
    result = func(head, k)
    after_nodes = nodes_of(result)
    assert [n.val for n in after_nodes] == expected, (values, k, to_list(result), expected)
    assert sorted(id(n) for n in after_nodes) == sorted(before), "phải dùng lại đúng các node cũ"
    assert all(before[id(n)] == n.val for n in after_nodes), "không được đổi value"


tests = [
    ([1, 2, 3, 4, 5], 2, [2, 1, 4, 3, 5]),          # ví dụ 1
    ([1, 2, 3, 4, 5], 3, [3, 2, 1, 4, 5]),          # ví dụ 2
    ([1], 1, [1]),                                  # n = 1
    ([1, 2, 3], 1, [1, 2, 3]),                      # k = 1: không đổi
    ([1, 2, 3], 3, [3, 2, 1]),                      # k = n: đảo toàn bộ
    ([1, 2, 3, 4, 5, 6], 3, [3, 2, 1, 6, 5, 4]),    # chia hết, không có phần thừa
    ([1, 2, 3, 4, 5, 6, 7, 8], 3, [3, 2, 1, 6, 5, 4, 7, 8]),
    ([5, 5, 0, 0], 2, [5, 5, 0, 0]),                # value trùng nhau
]

sol = Solution()
for values, k, expected in tests:
    check_solution(sol.reverseKGroup, values, k, expected)
    check_solution(reverse_k_group_iterative, values, k, expected)
    check_solution(reverse_k_group_stack, values, k, expected)
    check_solution(reverse_k_group_recursive, values, k, expected)
    print(f"{str(values):<26} k={k} -> {expected}  PASS")

# n lớn nhất theo constraints, k = 1 (cách đệ quy sẽ hỏng, cách lặp vẫn ổn)
big_values = [i % 1001 for i in range(5000)]
check_solution(sol.reverseKGroup, big_values, 1, big_values)
check_solution(sol.reverseKGroup, big_values, 5000, big_values[::-1])
print("n = 5000 PASS")

[1, 2, 3, 4, 5]            k=2 -> [2, 1, 4, 3, 5]  PASS
[1, 2, 3, 4, 5]            k=3 -> [3, 2, 1, 4, 5]  PASS
[1]                        k=1 -> [1]  PASS
[1, 2, 3]                  k=1 -> [1, 2, 3]  PASS
[1, 2, 3]                  k=3 -> [3, 2, 1]  PASS
[1, 2, 3, 4, 5, 6]         k=3 -> [3, 2, 1, 6, 5, 4]  PASS
[1, 2, 3, 4, 5, 6, 7, 8]   k=3 -> [3, 2, 1, 6, 5, 4, 7, 8]  PASS
[5, 5, 0, 0]               k=2 -> [5, 5, 0, 0]  PASS
n = 5000 PASS


### Đối chiếu chéo các cách trên input ngẫu nhiên

Dùng Cách 1 (đơn giản, dễ tin) làm "đáp án chuẩn", sinh ngẫu nhiên nhiều danh sách nhỏ với `random.seed` cố định để lần nào chạy cũng giống nhau.

In [23]:
import random

random.seed(25)
count = 0
for _ in range(500):
    n = random.randint(1, 12)
    k = random.randint(1, n)                   # đúng constraints: 1 <= k <= n
    values = [random.randint(0, 1000) for _ in range(n)]
    expected = to_list(reverse_k_group_values(build_list(values), k))
    for func in (reverse_k_group_stack, reverse_k_group_recursive,
                 reverse_k_group_iterative, sol.reverseKGroup):
        check_solution(func, values, k, expected)
    count += 1
print(f"{count} bộ test ngẫu nhiên: 4 cách khớp với baseline, đều dùng lại node cũ. PASS")

500 bộ test ngẫu nhiên: 4 cách khớp với baseline, đều dùng lại node cũ. PASS


## 10. Vì sao Cách 4 đúng?

**Bất biến (invariant)** ở đầu mỗi vòng `while True`:

> Đoạn từ `dummy.next` tới `group_prev` là kết quả **đã hoàn chỉnh** cho tất cả các nhóm đã xử lý, và `group_prev.next` là node đầu của **phần chưa xử lý** (vẫn nguyên thứ tự gốc).

- **Lúc đầu:** `group_prev = dummy`, phần đã xử lý rỗng, `dummy.next = head` là toàn bộ phần chưa xử lý. Đúng.
- **Một vòng có đủ `k` node:** vòng `prev/curr/nxt` lật đúng `k` mũi tên trong nhóm (dừng khi `curr` chạm `group_next`); nhờ `prev = group_next` ban đầu, đuôi mới `A` trỏ tới `group_next`. Sau đó `group_prev.next = kth` gắn nhóm đã đảo vào sau phần hoàn chỉnh, và `group_prev = A` là node cuối của phần hoàn chỉnh mới. Bất biến vẫn đúng.
- **Vòng không đủ `k` node:** `get_kth` trả `None` → dừng **trước khi đụng vào bất kỳ con trỏ nào**, nên phần thừa giữ nguyên thứ tự. Kết quả `dummy.next` đúng.
- **Dừng chắc chắn:** mỗi vòng `group_prev` tiến thêm đúng `k >= 1` node, danh sách hữu hạn.

Không có node mới nào được tạo (trừ `dummy`) và không dòng nào gán `.val` → đúng yêu cầu "chỉ đổi node".

## 11. So sánh các cách

Gọi `n` là số node, `k` là kích thước nhóm.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Đúng tinh thần đề? | Nhận xét |
|---|---|:---:|:---:|:---:|---|
| 1. Chép giá trị | list Python + slicing, dựng list mới | `O(n)` | `O(n)` | Không (tạo node mới / đổi value) | Chỉ để hiểu đề và làm baseline test |
| 2. Stack | push `k` node, pop ra theo thứ tự ngược | `O(n)` | `O(k)` | Có | Dễ nghĩ, nhưng `k` lớn thì tốn nhớ |
| 3. Đệ quy | đảo `k` node đầu, `head.next = đệ quy(phần sau)` | `O(n)` | `O(n/k)` call stack | Có | Code gọn; `k` nhỏ + `n` lớn dễ `RecursionError` |
| 4. Lặp `dummy` + `group_prev` | `get_kth` → đảo tại chỗ với `prev = group_next` → nối | `O(n)` | `O(1)` | Có | Đáp án follow-up, nên dùng khi phỏng vấn |

Về thời gian Cách 4: mỗi node bị `get_kth` đi qua 1 lần và bị đảo 1 lần, cộng thêm tối đa `k` bước kiểm tra cho nhóm thiếu cuối cùng → khoảng `2n` bước, tức `O(n)`.

## 12. Các lỗi người mới hay gặp

### Lỗi 1: Lật mũi tên trước khi lưu `nxt`

Sai:

```python
curr.next = prev
prev = curr
curr = curr.next     # curr.next vừa bị đổi thành prev!
```

Sau dòng 1, đường đi tới phần còn lại đã **mất**: `curr.next` giờ là `prev` cũ (ở vòng đầu là `None`), nên `curr = curr.next` không đi tiếp được. Ở vòng đầu, `curr` thành `None` → vòng lặp dừng ngay, các node `2, 3, 4` bị bỏ rơi. Hàm dưới đây (có thêm giới hạn bước cho an toàn) cho thấy hậu quả:

In [24]:
def reverse_list_wrong(head, max_steps=10):
    prev, curr = None, head
    steps = 0
    while curr is not None and steps < max_steps:
        curr.next = prev
        prev = curr
        curr = curr.next        # SAI: đáng lẽ phải lưu nxt từ trước
        steps += 1
    return prev


print("Sai:", show(reverse_list_wrong(build_list([1, 2, 3, 4]))))   # mất 2, 3, 4
print("Đúng:", show(reverse_list(build_list([1, 2, 3, 4]))))

Sai: 1 -> None
Đúng: 4 -> 3 -> 2 -> 1 -> None


### Lỗi 2: Đảo luôn nhóm cuối thiếu `k` node

Nếu chỉ "đảo tối đa `k` node" mà **không kiểm tra đủ `k` trước**, nhóm thừa cuối cũng bị đảo. Ví dụ đề 1 sẽ ra `[2,1,4,3,5]` (may mắn trùng vì nhóm cuối chỉ có 1 node!), nhưng ví dụ 2 thì sai:

In [25]:
def reverse_k_group_no_check(head, k):
    # SAI: không kiểm tra nhóm có đủ k node hay không
    if head is None:
        return None
    prev, curr, count = None, head, 0
    while curr is not None and count < k:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
        count += 1
    head.next = reverse_k_group_no_check(curr, k)
    return prev


print("k=2:", to_list(reverse_k_group_no_check(build_list([1, 2, 3, 4, 5]), 2)), "(trùng hợp đúng)")
print("k=3:", to_list(reverse_k_group_no_check(build_list([1, 2, 3, 4, 5]), 3)), "(sai, đúng phải là [3, 2, 1, 4, 5])")

k=2: [2, 1, 4, 3, 5] (trùng hợp đúng)
k=3: [3, 2, 1, 5, 4] (sai, đúng phải là [3, 2, 1, 4, 5])


**Sửa:** luôn kiểm tra trước (`get_kth` ở Cách 4, vòng đếm `count` ở Cách 3). Bài học test: ví dụ "có vẻ pass" chưa chắc đã đúng — luôn thử nhóm cuối có **từ 2 node trở lên** nhưng **ít hơn `k`**.

### Lỗi 3: Quên nối `group_prev.next = kth` hoặc quên dời `group_prev`

- Quên `group_prev.next = kth`: node đứng trước vẫn trỏ vào node đầu cũ (giờ là đuôi nhóm) → **mất** các node khác của nhóm.
- Quên `group_prev = old_first`: vòng sau lại lấy `dummy` làm node đứng trước → đảo đi đảo lại chính nhóm đầu.
- Viết `group_prev = kth` (nhầm đầu mới với đuôi mới): `kth` giờ đứng **đầu** nhóm, đi `k` bước từ nó sẽ lấy sai nhóm.

Mẹo nhớ: **node đầu cũ = đuôi mới = `group_prev` mới.** Phải lưu nó (`old_first = group_prev.next`) **trước** khi gán `group_prev.next = kth`.

In [26]:
def reverse_k_group_forget_link(head, k):
    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        kth = get_kth(group_prev, k)
        if kth is None:
            break
        group_next = kth.next
        prev, curr = group_next, group_prev.next
        while curr is not group_next:
            nxt = curr.next
            curr.next = prev
            prev = curr
            curr = nxt
        old_first = group_prev.next
        # QUÊN: group_prev.next = kth
        group_prev = old_first
    return dummy.next


print("Sai :", show(reverse_k_group_forget_link(build_list([1, 2, 3, 4, 5, 6, 7, 8]), 3)))
print("Đúng:", show(reverse_k_group_iterative(build_list([1, 2, 3, 4, 5, 6, 7, 8]), 3)))

Sai : 1 -> 4 -> 7 -> 8 -> None
Đúng: 3 -> 2 -> 1 -> 6 -> 5 -> 4 -> 7 -> 8 -> None


Giải thích output sai: `dummy` vẫn trỏ vào `1` (giờ là đuôi nhóm 1, trỏ sang `4`), node `1` vẫn trỏ vào `4` (giờ là đuôi nhóm 2, trỏ sang `7`). Kết quả chỉ còn `1 -> 4 -> 7 -> 8`, các node `3, 2, 6, 5` bị "lạc" khỏi danh sách.

### Lỗi 4: Khởi tạo `prev = None` khi đảo nhóm ở Cách 4

Nếu dùng `prev = None` thì đuôi nhóm trỏ `None` → danh sách bị **cắt đứt** sau nhóm đầu. Phải dùng `prev = group_next`, hoặc nối lại thủ công `old_first.next = group_next`.

### Lỗi 5: Đổi `val` cho "nhanh"

Swap giá trị giữa các node cho ra đúng output, nhưng vi phạm đề ("only nodes themselves may be changed"). Trong phỏng vấn sẽ bị đánh trượt.

### Lỗi 6: Dùng `==` thay cho `is` khi so node

`while curr != group_next` vẫn chạy với `ListNode` chưa định nghĩa `__eq__` (lúc đó `!=` so object), nhưng `is` / `is not` diễn đạt đúng ý "cùng một node" và không bị ảnh hưởng nếu ai đó định nghĩa `__eq__` theo value.

## 13. Mẹo nhận dạng pattern

- Đề có **linked list** + "đảo / xoay / hoán đổi một đoạn" → nghĩ ngay tới bộ 4 dòng `nxt = curr.next; curr.next = prev; prev = curr; curr = nxt`.
- Đầu danh sách **có thể thay đổi** → dùng **dummy node**, trả `dummy.next`.
- Xử lý theo **từng khúc** → giữ con trỏ "node đứng trước khúc" (`group_prev`) và "node đầu khúc sau" (`group_next`); trước khi đảo, **nhìn trước** xem khúc có đủ không (`get_kth`).
- Đề hỏi bộ nhớ `O(1)` → tránh stack/list phụ và tránh đệ quy sâu; ưu tiên vòng lặp với vài con trỏ.
- Luôn vẽ hình: mỗi lần gán `x.next = ...` là **xóa một mũi tên cũ** — hỏi "mũi tên cũ đó dẫn tới đâu, mình đã lưu chưa?".

## 14. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| 206. Reverse Linked List | Nền tảng `prev/curr/nxt` (mục 3) |
| 24. Swap Nodes in Pairs | Chính là bài 25 với `k = 2` |
| 92. Reverse Linked List II | Đảo đoạn từ vị trí `left` đến `right`: dùng `dummy` + node đứng trước đoạn |
| 2074. Reverse Nodes in Even Length Groups | Nhóm có độ dài 1, 2, 3, ...; chỉ đảo nhóm có độ dài chẵn |
| 234. Palindrome Linked List | Đảo nửa sau danh sách rồi so sánh |
| 61. Rotate List | Nối lại con trỏ theo khúc, cần tìm node đứng trước điểm cắt |

## 15. Bài tập vận dụng

Làm từ dễ tới khó. Các cell TODO chỉ định nghĩa hàm, phần test đang để comment — làm xong thì bỏ comment để tự kiểm tra. Đáp án ở **cuối notebook**.

### Bài 1 — Đoán output (không chạy code)

Với `head = [1,2,3,4,5,6]`, kết quả của `reverseKGroup(head, 4)` là gì? Còn `k = 6`? Còn `k = 5`?

### Bài 2 — Tính tay bảng con trỏ

Chạy tay `reverse_list` trên `[10, 20, 30]`. Lập bảng: sau mỗi vòng, `prev` đang là danh sách nào, `curr` đang là danh sách nào?

| Vòng | `prev` | `curr` |
|:---:|---|---|
| bắt đầu | `None` | `10 -> 20 -> 30 -> None` |
| 1 | ? | ? |
| 2 | ? | ? |
| 3 | ? | ? |

### Bài 3 — Điền TODO: đảo ngược cả linked list

Hoàn thiện 4 dòng trong vòng lặp. Không nhìn lại mục 3 nếu có thể!

In [27]:
def exercise_reverse_list(head: Optional[ListNode]) -> Optional[ListNode]:
    prev = None
    curr = head
    while curr is not None:
        # TODO 1: lưu node kế tiếp
        # TODO 2: lật mũi tên của curr
        # TODO 3: dời prev
        # TODO 4: dời curr
        pass
    return prev

# Bỏ comment sau khi làm xong:
# assert to_list(exercise_reverse_list(build_list([1, 2, 3]))) == [3, 2, 1]
# assert to_list(exercise_reverse_list(build_list([7]))) == [7]
# assert exercise_reverse_list(None) is None

### Bài 4 — Tự viết lại: Swap Nodes in Pairs (LeetCode 24)

Hoán đổi từng cặp node kề nhau: `[1,2,3,4,5]` → `[2,1,4,3,5]`. Không đổi value, **không gọi lại** các hàm `reverse_k_group_*` — tự viết với `dummy` + một con trỏ `prev` đứng trước cặp.

In [28]:
def exercise_swap_pairs(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO: dummy -> ... ; với mỗi cặp (first, second): prev.next = second,
    #       first.next = second.next, second.next = first, prev = first
    raise NotImplementedError

# Bỏ comment sau khi làm xong:
# assert to_list(exercise_swap_pairs(build_list([1, 2, 3, 4]))) == [2, 1, 4, 3]
# assert to_list(exercise_swap_pairs(build_list([1, 2, 3, 4, 5]))) == [2, 1, 4, 3, 5]
# assert to_list(exercise_swap_pairs(build_list([1]))) == [1]
# assert exercise_swap_pairs(None) is None

### Bài 5 — Biến thể: đảo cả nhóm cuối dù thiếu

Đổi luật: nhóm cuối **thiếu `k` node cũng bị đảo**. Ví dụ `[1,2,3,4,5]`, `k = 3` → `[3,2,1,5,4]`. Hãy sửa Cách 4 (bộ nhớ `O(1)`).

In [29]:
def exercise_reverse_all_groups(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    # TODO: giống Cách 4, nhưng khi không đủ k node thì vẫn đảo phần còn lại
    raise NotImplementedError

# Bỏ comment sau khi làm xong:
# assert to_list(exercise_reverse_all_groups(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 5, 4]
# assert to_list(exercise_reverse_all_groups(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
# assert to_list(exercise_reverse_all_groups(build_list([1, 2]), 5)) == [2, 1]

### Bài 6 — Mini challenge: Reverse Linked List II (LeetCode 92)

Đảo đoạn từ vị trí `left` tới `right` (đánh số từ 1), một lần duyệt, bộ nhớ `O(1)`. Ví dụ `[1,2,3,4,5]`, `left = 2`, `right = 4` → `[1,4,3,2,5]`.

In [30]:
def exercise_reverse_between(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    # TODO: dummy; tìm node đứng trước vị trí left; đảo (right - left + 1) node; nối lại hai đầu
    raise NotImplementedError

# Bỏ comment sau khi làm xong:
# assert to_list(exercise_reverse_between(build_list([1, 2, 3, 4, 5]), 2, 4)) == [1, 4, 3, 2, 5]
# assert to_list(exercise_reverse_between(build_list([5]), 1, 1)) == [5]
# assert to_list(exercise_reverse_between(build_list([1, 2, 3]), 1, 3)) == [3, 2, 1]

## 16. Gợi ý theo mức

**Bài 1**
- Mức 1: chia nhóm 4 → `[1,2,3,4] [5,6]`; nhóm nào đủ?
- Mức 2: `k = 6` bằng `n` → đảo toàn bộ; `k = 5` → một nhóm 5 + một node thừa.

**Bài 2**
- Mức 1: `prev` luôn là phần đã đảo (đọc từ node mới nhất về `None`).
- Mức 2: sau vòng `i`, `prev` chứa `i` node đầu theo thứ tự ngược; `curr` là phần còn lại.

**Bài 3**
- Mức 1: phải giữ đường đi tiếp trước khi đổi `curr.next`.
- Mức 2: thứ tự: lưu → lật → dời `prev` → dời `curr`.
- Mức 3: `nxt = curr.next`, `curr.next = prev`, `prev = curr`, `curr = nxt`.

**Bài 4**
- Mức 1: một cặp cần 3 node: node đứng trước (`prev`), `first`, `second`.
- Mức 2: vòng lặp chạy khi `prev.next` và `prev.next.next` đều khác `None`.
- Mức 3: gán theo thứ tự `first.next = second.next`, `second.next = first`, `prev.next = second`, rồi `prev = first`.

**Bài 5**
- Mức 1: không cần `get_kth` để quyết định dừng nữa; chỉ cần biết nhóm kết thúc ở đâu.
- Mức 2: đi tối đa `k` bước để tìm `group_next` (có thể là `None`); đảo tới khi `curr is group_next`.
- Mức 3: sau khi đảo, nhớ `old_first`, gán `group_prev.next = prev` (đầu mới là `prev`, không phải `kth`), dừng khi `group_next is None`.

**Bài 6**
- Mức 1: đây là Cách 4 với đúng **một nhóm**, `group_prev` là node ở vị trí `left - 1`.
- Mức 2: từ `dummy` đi `left - 1` bước được `group_prev`; từ `group_prev` đi `right - left + 1` bước được node cuối đoạn.
- Mức 3: đảo với `prev = group_next`, rồi `group_prev.next = node cuối đoạn`.

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:** `k = 4` → `[4,3,2,1,5,6]`; `k = 6` → `[6,5,4,3,2,1]`; `k = 5` → `[5,4,3,2,1,6]`.

**Bài 2:**

| Vòng | `prev` | `curr` |
|:---:|---|---|
| bắt đầu | `None` | `10 -> 20 -> 30 -> None` |
| 1 | `10 -> None` | `20 -> 30 -> None` |
| 2 | `20 -> 10 -> None` | `30 -> None` |
| 3 | `30 -> 20 -> 10 -> None` | `None` |

Cell dưới kiểm chứng Bài 1, Bài 2 bằng code, sau đó là đáp án Bài 3–6 có assert.

In [31]:
# Kiểm chứng Bài 1
for k in (4, 6, 5):
    print(f"k = {k}:", to_list(Solution().reverseKGroup(build_list([1, 2, 3, 4, 5, 6]), k)))
assert to_list(Solution().reverseKGroup(build_list([1, 2, 3, 4, 5, 6]), 4)) == [4, 3, 2, 1, 5, 6]

# Kiểm chứng Bài 2
_ = trace_reverse_list([10, 20, 30])

k = 4: [4, 3, 2, 1, 5, 6]
k = 6: [6, 5, 4, 3, 2, 1]
k = 5: [5, 4, 3, 2, 1, 6]
bắt đầu | prev: None                   | curr: 10 -> 20 -> 30 -> None
vòng 1  | prev: 10 -> None             | curr: 20 -> 30 -> None
vòng 2  | prev: 20 -> 10 -> None       | curr: 30 -> None
vòng 3  | prev: 30 -> 20 -> 10 -> None | curr: None


In [32]:
def answer_reverse_list(head):
    prev = None
    curr = head
    while curr is not None:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    return prev


def answer_swap_pairs(head):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next is not None and prev.next.next is not None:
        first = prev.next
        second = first.next
        first.next = second.next
        second.next = first
        prev.next = second
        prev = first
    return dummy.next


def answer_reverse_all_groups(head, k):
    dummy = ListNode(0, head)
    group_prev = dummy
    while group_prev.next is not None:
        # tìm group_next: đi tối đa k bước từ đầu nhóm
        group_next = group_prev.next
        for _ in range(k):
            if group_next is None:
                break
            group_next = group_next.next
        # đảo nhóm (có thể thiếu k node)
        prev, curr = group_next, group_prev.next
        while curr is not group_next:
            nxt = curr.next
            curr.next = prev
            prev = curr
            curr = nxt
        old_first = group_prev.next
        group_prev.next = prev          # prev là đầu mới của nhóm
        group_prev = old_first
    return dummy.next


def answer_reverse_between(head, left, right):
    dummy = ListNode(0, head)
    group_prev = dummy
    for _ in range(left - 1):
        group_prev = group_prev.next
    last = group_prev
    for _ in range(right - left + 1):
        last = last.next
    group_next = last.next
    prev, curr = group_next, group_prev.next
    while curr is not group_next:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    group_prev.next = last
    return dummy.next


assert to_list(answer_reverse_list(build_list([1, 2, 3]))) == [3, 2, 1]
assert to_list(answer_reverse_list(build_list([7]))) == [7]
assert answer_reverse_list(None) is None

assert to_list(answer_swap_pairs(build_list([1, 2, 3, 4]))) == [2, 1, 4, 3]
assert to_list(answer_swap_pairs(build_list([1, 2, 3, 4, 5]))) == [2, 1, 4, 3, 5]
assert to_list(answer_swap_pairs(build_list([1]))) == [1]
assert answer_swap_pairs(None) is None

assert to_list(answer_reverse_all_groups(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 5, 4]
assert to_list(answer_reverse_all_groups(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(answer_reverse_all_groups(build_list([1, 2]), 5)) == [2, 1]

assert to_list(answer_reverse_between(build_list([1, 2, 3, 4, 5]), 2, 4)) == [1, 4, 3, 2, 5]
assert to_list(answer_reverse_between(build_list([5]), 1, 1)) == [5]
assert to_list(answer_reverse_between(build_list([1, 2, 3]), 1, 3)) == [3, 2, 1]

# Swap pairs phải khớp reverseKGroup với k = 2
random.seed(24)
for _ in range(200):
    values = [random.randint(0, 9) for _ in range(random.randint(0, 10))]
    expected = to_list(reverse_k_group_values(build_list(values), 2)) if values else []
    assert to_list(answer_swap_pairs(build_list(values))) == expected

print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 18. Template cuối cùng nên nhớ

```python
class Solution:
    def reverseKGroup(self, head: Optional[ListNode], k: int) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        group_prev = dummy
        while True:
            kth = group_prev                 # 1. nhìn trước k bước
            for _ in range(k):
                kth = kth.next
                if kth is None:
                    return dummy.next        #    thiếu -> giữ nguyên, xong
            group_next = kth.next

            prev, curr = group_next, group_prev.next   # 2. đảo tại chỗ
            while curr is not group_next:
                nxt = curr.next
                curr.next = prev
                prev = curr
                curr = nxt

            old_first = group_prev.next      # 3. nối lại + dời
            group_prev.next = kth
            group_prev = old_first
```

Câu thần chú: **"Nhìn trước k bước → đảo với `prev = group_next` → nối `group_prev` vào `kth` → `group_prev` = đầu cũ."**

## 19. Checklist tự kiểm tra trước khi submit

- Mình có dùng **dummy node** để không phải xử lý riêng nhóm đầu tiên chưa?
- Mình có **kiểm tra đủ `k` node trước khi đảo** không (nhóm cuối thiếu phải giữ nguyên)?
- Trong vòng đảo, mình có **lưu `nxt` trước** khi gán `curr.next` không?
- Đuôi nhóm có được nối sang `group_next` không (`prev = group_next` hoặc nối thủ công)?
- Mình có lưu `old_first` **trước** khi gán `group_prev.next = kth`, và dời `group_prev = old_first` không?
- Có tạo node mới hay đổi `val` không? (Không được!)
- Đã thử: `k = 1`, `k = n`, `n = 1`, `n` chia hết cho `k`, nhóm cuối thiếu từ 2 node trở lên chưa?
- Bộ nhớ có thật sự `O(1)` không (không stack, không đệ quy)?